# Worst-of autocallables against their disclosed estimated values

This notebook presents the finished results and nothing else. It reads
`data/processed/` and recomputes nothing, so every number here matches the run
that produced it and the tables in `reports/`. Run the pipeline first:

```bash
make surfaces validate analysis
```

Cells run top to bottom with no hidden state.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# The five-year notes take about half an hour each, so they were run into a
# separate results file. load_results merges the two and hands back one set.
from src.analysis.results import load_results

results, meta = load_results(ROOT)
surfaces = json.loads((ROOT / "data/processed/surfaces.json").read_text())
notes = {n["id"]: n for n in yaml.safe_load((ROOT / "config/notes.yaml").read_text())["notes"]}
analysis = {"notes": results, "generated_for_snapshot": meta.get("snapshot"),
            "convergence": meta.get("convergence") or {}}

pd.set_option("display.width", 160)
print(f"option snapshot {analysis['generated_for_snapshot']}, {len(results)} notes priced")


## 1. The surfaces the pricing rests on

Before any note value means anything, the surfaces have to be arbitrage-free and
the local-vol engine has to reproduce the vanilla prices it was calibrated to.
Both are checked here rather than asserted.

In [ ]:
rows = []
for ticker, record in surfaces.items():
    errors = np.array([s["rmse_vol"] for s in record["slices"]])
    rows.append({
        "underlying": ticker,
        "chain": record["chain_symbol"],
        "expiries": len(record["slices"]),
        "longest (y)": round(max(s["t"] for s in record["slices"]), 2),
        "fit median (bp)": round(float(np.median(errors)) * 1e4, 1),
        "fit worst (bp)": round(float(errors.max()) * 1e4, 1),
        "butterfly fails": record["audit"]["butterfly_failures"],
        "calendar fails": record["audit"]["calendar_failures"],
    })
pd.DataFrame(rows)

In [ ]:
# The repricing gate, in implied-vol points so the five names are comparable.
# SPX is the one that misses its quoted spread, and the spread column explains why.
pd.read_csv(ROOT / "data/processed/reprice_volpoints.csv")

## 2. Model values against the disclosed values

The headline result. Every model value sits below the issuer's own number, and
the shortfall grows with tenor.

In [ ]:
summary = pd.DataFrame([
    {
        "note": note_id,
        "issuer": result["base"]["issuer"].split()[0],
        "tenor": round(result["base"]["tenor_years"], 2),
        "coupon": f"{result['base']['coupon_rate']*100:.2f}%",
        "call": result["base"]["call_style"],
        "model": round(result["base"]["value"], 2),
        "s.e.": round(result["base"]["stderr"], 2),
        "disclosed": result["margin"]["disclosed_estimated_value"],
        "gap": round(result["base"]["value"] - result["margin"]["disclosed_estimated_value"], 2),
    }
    for note_id, result in results.items()
])
summary

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.scatter(summary["tenor"], summary["gap"], s=55, zorder=3)
for row in summary.itertuples():
    ax.annotate(f"{row.note} {row.issuer}", (row.tenor, row.gap),
                textcoords="offset points", xytext=(6, -3), fontsize=8)
ax.axhline(0, color="k", lw=0.9)
ax.set_xlabel("Tenor, years")
ax.set_ylabel("Model minus disclosed, $ per $1,000")
ax.set_title("The shortfall to the disclosed value widens with tenor")
ax.grid(alpha=0.25)
plt.tight_layout()

## 3. Why no funding spread explains it

A funding spread discounts the note's cashflows more heavily, so it can only
push a model value **down**. A model already below the disclosed value therefore
needs a negative spread, which is not a funding rate.

For an issuer-callable note there is a second effect, and it is the one that
settles the Goldman note: as the discount rate falls, the note becomes more
valuable to hold, so a rational issuer calls it sooner. That caps the value near
par at any spread.

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 4.4))
for note_id, result in results.items():
    curve = result.get("value_vs_spread") or []
    if not curve:
        continue
    line, = ax.plot([p["spread_bp"] for p in curve], [p["value"] for p in curve],
                    marker="o", ms=3.5, lw=1.4, label=note_id)
    ax.axhline(result["margin"]["disclosed_estimated_value"],
               color=line.get_color(), lw=0.8, ls=":", alpha=0.8)
ax.axvline(0, color="k", lw=0.9)
ax.set_xlabel("Funding spread over Treasuries, bp")
ax.set_ylabel("Model value per $1,000")
ax.set_title("Dotted lines are the disclosed values: they sit above every curve")
ax.legend(fontsize=8, ncol=3)
ax.grid(alpha=0.25)
plt.tight_layout()

In [ ]:
benchmark = next(iter(results.values()))["corporate_oas_benchmark"]
print(f"benchmark: {benchmark['description']}")
print(f"           {benchmark['oas_bp']:.0f} bp on {benchmark['as_of']}\n")

spreads = []
for note_id, result in results.items():
    optimal = result["implied_spread_optimal_exercise"]
    no_call = result["implied_spread_no_early_call"]
    fmt = lambda s: ("no solution" if s is None or not s["bracketed"]
                     else f"{s['spread']*1e4:+.0f} bp")
    spreads.append({
        "note": note_id,
        "optimal exercise": fmt(optimal),
        "call ignored": "not applicable" if no_call is None else fmt(no_call),
    })
pd.DataFrame(spreads)

## 4. What the issuer keeps, with no model involved

The one margin figure that carries no modelling assumption: the issuer's own
arithmetic from its own filing.

In [ ]:
pd.DataFrame([
    {
        "note": note_id,
        "issuer": result["margin"]["issuer"].split()[0],
        "disclosed": result["margin"]["disclosed_estimated_value"],
        "commission": result["margin"]["selling_commission"],
        "1000 - value": round(result["margin"]["ev_gap"], 2),
        "residual": round(result["margin"]["residual_margin"], 2),
        "residual %": round(result["margin"]["residual_margin_pct"], 2),
    }
    for note_id, result in results.items()
])

## 5. Risk

The correlation sign is the one the brief says to confirm numerically rather
than reason out. Lower correlation raises the chance that at least one
underlying breaches its barrier, which enriches the down-and-in put the investor
is short, so the investor is **long** correlation.

In [ ]:
risk = []
for note_id, result in results.items():
    correlation = result["correlation"]
    risk.append({
        "note": note_id,
        **{f"delta {d['underlying']}": round(d["delta_per_1pct"], 2) for d in result["deltas"]},
        "total vega": round(sum(v["vega_per_vol_point"] for v in result["bucketed_vega"]), 2),
        "d/d corr (+0.05)": round(correlation["d_value_per_0.05"], 2),
        "long correlation": correlation["investor_is_long_correlation"],
    })
pd.DataFrame(risk)

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 4.0))
buckets = [b["bucket_years"] for b in next(iter(results.values()))["bucketed_vega"]]
width = 0.8 / len(results)
for offset, (note_id, result) in enumerate(results.items()):
    ax.bar(np.arange(len(buckets)) + offset * width - 0.4,
           [b["vega_per_vol_point"] for b in result["bucketed_vega"]],
           width=width, label=note_id)
ax.axhline(0, color="k", lw=0.9)
ax.set_xticks(np.arange(len(buckets)))
ax.set_xticklabels(buckets)
ax.set_xlabel("Maturity bucket, years")
ax.set_ylabel("$ per vol point")
ax.set_title("Vega sits where the note is still likely to be alive")
ax.legend(fontsize=8, ncol=3)
ax.grid(alpha=0.25)
plt.tight_layout()

## 6. Model risk

Three questions, each answered by repricing the same note under a different
assumption: how much the value depends on the smile rather than the vol level,
how much the long-dated extrapolation is worth, and what the two-day gap between
pricing date and option snapshot costs.

In [ ]:
pd.DataFrame([
    {
        "note": note_id,
        "local vol": round(result["flat_vol"]["local_vol_value"], 2),
        "flat vol": round(result["flat_vol"]["flat_vol_value"], 2),
        "smile effect": round(result["flat_vol"]["difference"], 2),
        "extrapolation effect": round(result["extrapolation"]["difference"], 2),
        "timing effect": round(result["timing"]["difference"], 2),
    }
    for note_id, result in results.items()
])

In [ ]:
# How far past the longest listed expiry each note reaches, per underlying.
reliance = []
for note_id, result in results.items():
    for ticker, detail in result["extrapolation"]["extrapolation_reliance"].items():
        reliance.append({"note": note_id, "underlying": ticker, **detail})
pd.DataFrame(reliance).pivot(index="note", columns="underlying",
                             values="extrapolated_years").fillna("")

## 7. Has the simulation converged?

Price and standard error against path count, with the control variates in place.

In [ ]:
convergence = analysis.get("convergence") or {}
if convergence:
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
    for note_id, rows in sorted(convergence.items()):
        paths = [r["paths"] for r in rows]
        axes[0].errorbar(paths, [r["value"] for r in rows],
                         yerr=[2 * r["stderr"] for r in rows],
                         marker="o", ms=4, capsize=3, label=note_id)
        axes[1].loglog(paths, [r["stderr"] for r in rows], marker="o", ms=4, label=note_id)
    axes[0].set_xscale("log")
    axes[0].set_xlabel("paths"); axes[0].set_ylabel("value per $1,000")
    axes[0].set_title("Price against path count (bars: two standard errors)")
    axes[1].set_xlabel("paths"); axes[1].set_ylabel("standard error, $")
    axes[1].set_title("Standard error falls at the Monte Carlo rate")
    for axis in axes:
        axis.legend(fontsize=8); axis.grid(alpha=0.25)
    plt.tight_layout()
else:
    print("convergence study not present; run the analysis without --no-convergence")

In [ ]:
# Variance reduction delivered by the European control variates.
pd.DataFrame([
    {
        "note": note_id,
        "s.e. plain": round(result["base"]["stderr_no_control_variate"], 3),
        "s.e. with controls": round(result["base"]["stderr"], 3),
        "variance reduction": f"{result['base']['variance_reduction_factor']:.2f}x",
    }
    for note_id, result in results.items()
])